In [ ]:
from ngsolve import *
from netgen.geom2d import *
from netgen.occ import *
from ngsolve.webgui import Draw
import numpy as np

# ------------------------------------------------------------
# Parameter
# ------------------------------------------------------------
H, L = 4.0, 28.0
cx, cy, R = 7.0, 0.0, 0.5
# H, L = 0.41, 2.0
# cx, cy, R = 0.5, 0.0, 0.05

nu = 1e-3       #viscosity

# ------------------------------------------------------------
# Geometrie
# Rectangle(width, height) liegt standardmäßig in [0,L]×[0,H]
# → daher zuerst erzeugen, dann verschieben
# ------------------------------------------------------------
rect = MoveTo(0,-H/2).Rectangle(L, H).Face()

rect.edges.Min(X).name = "inlet"
rect.edges.Max(X).name = "outlet"
rect.edges.Min(Y).name = "walls"
rect.edges.Max(Y).name = "walls"

cyl = Circle((cx,cy), R).Face()
cyl.edges.name = "obstacle"

shape = rect - cyl
#Draw(shape)

# ------------------------------------------------------------
# Mesh
# ------------------------------------------------------------
mesh = Mesh(OCCGeometry(shape,dim=2).GenerateMesh(maxh=0.4))
#mesh.Refine()
Draw(mesh);

WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

In [2]:
# ------------------------------------------------------------
# FE-Räume (Taylor–Hood)
# ------------------------------------------------------------
V = VectorH1(mesh, order=2, dirichlet="inlet|walls|obstacle")
Q = H1(mesh, order=1, dirichlet="outlet")   # Variante C: integral(p)=0

X = FESpace([V, Q])
(u, p) = X.TrialFunction()
(v, q) = X.TestFunction()

gfu = GridFunction(X)
gfu_u, gfu_p = gfu.components


In [ ]:
# ------------------------------------------------------------
# Inlet-Profil (parabolisch auf [-H/2, H/2])            -> Randbedingungen bei Inlet:  u = uin bei "inlet"
# ------------------------------------------------------------
Umax = 4.              #Reynoldszahl =150 = Umax*D/viskosität = 0.5 * 0.5/1e-3 = 250


uin_x = Umax*(1 - ((2*y)/H)**2)   # Maximum bei y=0 
#uin_x = (1.5*4*y*(y)/(0.41*0.41))



eps = 1e-5
uin = CoefficientFunction((uin_x, 0))

gfu_u.Set(uin, definedon=mesh.Boundaries("inlet"))


In [4]:
# Reynolds Zahl ausgeben
Reynold = Umax * R/nu
print("Reynolds-Zahl = ",Reynold)

Reynolds-Zahl =  200.0


In [5]:
# ------------------------------------------------------------
# Stokes: schwache Form
# ------------------------------------------------------------

#für stabilität
alpha = 1e-6
gamma = 0


a = BilinearForm(X)
a += nu*InnerProduct(Grad(u), Grad(v)) * dx         #TODO Sym weglassen
a += gamma * div(u) * div(v) * dx
a += -div(v)*p * dx
a += -div(u)*q * dx
a += alpha * p*q * dx

L = LinearForm(X)   # keine Volumenkräfte

a.Assemble()
L.Assemble()

# ------------------------------------------------------------
# Lösen -- hatte starke Probleme beim lösen!!!
# ------------------------------------------------------------

inv_stokes = a.mat.Inverse(X.FreeDofs())                #TODO

res = L.vec - a.mat*gfu.vec
gfu.vec.data += inv_stokes * res

#Draw (gfu.components[0], mesh);
# ------------------------------------------------------------
# Visualisierung
# ------------------------------------------------------------


Draw(gfu_u, mesh, "velocity")
#Draw(gfu_p, mesh, "pressure")


# Stokes hier fertig gelöst -> jetzt gehts an NavierStokes


WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

BaseWebGuiScene

In [6]:
def ApplyDirichlet(gf_u, gf_p):
    # Geschwindigkeit: inlet vorgeben, Wände+Zylinder no-slip
    gf_u.Set(uin, definedon=mesh.Boundaries("inlet"))
    gf_u.Set(CoefficientFunction((0,0)), definedon=mesh.Boundaries("walls|obstacle"))
    # Druckreferenz: p=0 am outlet
    #gf_p.Set(0, definedon=mesh.Boundaries("outlet"))

# -----------------------------
# 2) Navier–Stokes via Picard(Oseen) pro Zeitschritt
#     Fixpunkt: w^k = u^k
# -----------------------------

print("Reynolds-Zahl=",Reynold)

t = 0.0
i = 0
dt   = 0.01
tend = 10.0
picard_maxit = 3
picard_tol   = 1e-6

underrelaxation_factor = 1      # u_k+1 = u_k + x*A_inv(f-A*u_k)

#für endsimulation
gfut = GridFunction(V, multidim=0)
vel = gfu.components[0]

scene = Draw(gfu_u,mesh)

while t < tend - 1e-12:

    # u^n
    u_prev = GridFunction(V)
    u_prev.vec.data = gfu_u.vec

    # Start für Picard: u^{n+1,0} := u^n
    gfu_new = GridFunction(X)
    gfu_new.components[0].vec.data = u_prev.vec
    gfu_new.components[1].vec.data = gfu_p.vec
    ApplyDirichlet(gfu_new.components[0], gfu_new.components[1])

    # Konvektionsfeld w^0
    w = GridFunction(V)
    w.vec.data = u_prev.vec

    for k in range(picard_maxit):
        w_old = GridFunction(V)
        w_old.vec.data = w.vec

        a = BilinearForm(X, symmetric=False)
        # Zeit: (1/dt)(u^{n+1},v)
        a += (1/dt)*InnerProduct(u, v)*dx
        # Viskosität
        a += nu*InnerProduct(Grad(u), Grad(v))*dx
        # Oseen-Konvektion: ((w·∇)u, v) = (Grad(u)*w, v)
        a += InnerProduct(Grad(u)*w, v)*dx
        # Inkompressibilität (Vorzeichen wie Stokes!)
        a += -div(v)*p*dx
        a += -div(u)*q*dx
        a += alpha * p*q * dx


        f = LinearForm(X)
        # RHS: (1/dt)(u^n, v)
        f += (1/dt)*InnerProduct(u_prev, v)*dx

        a.Assemble()
        f.Assemble()

        # Dirichlet-Werte müssen VOR dem Solve gesetzt sein
        ApplyDirichlet(gfu_new.components[0], gfu_new.components[1])

        # Lösen mit Elimination: erst Residuum, dann Korrektur auf FreeDofs
        inv = a.mat.Inverse(X.FreeDofs())
        res = f.vec - a.mat * gfu_new.vec
        gfu_new.vec.data += underrelaxation_factor * inv * res

        #gfu_new.vec.data = inv*f.vec       #wäre der mathematische Weg: Au = F <=> u = A^-1*f, aber nicht sehr stabil

        # Fixpunkt-Update
        w.vec.data = gfu_new.components[0].vec

        # Konvergenztest
        diff = Norm(w.vec - w_old.vec)       
        if diff < picard_tol:
            break

    # Zeitschritt übernehmen
    gfu.vec.data = gfu_new.vec
    t += dt
    i += 1

    if i%10 == 0: scene.Redraw()
    if i%50 == 0: 
        gfut.AddMultiDimComponent(vel.vec)
        # print(f"t = {t}", end='\r')

    #print(Norm(gfu_new.vec))


    # scene.Redraw()

# Ergebnis
Draw(gfu.components[0], mesh, "u_ns")
# Draw(gfu.components[1], mesh, "p_ns")


Reynolds-Zahl= 200.0


WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

BaseWebGuiScene

In [7]:
Draw (gfut, mesh, interpolate_multidim=True, animate=True);

WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…